# Multi-Seed Training for Statistical Significance

This notebook trains each algorithm variant **5 times** with different random seeds on the **11x11 grid** to produce training curves with mean ± std error bands.

### Models to train (11x11):
| # | Model | Mask | Approx Time |
|---|-------|------|--------------|
| 1 | Masked DRQN | Yes | ~5× single run |
| 2 | DRQN No Mask | No | ~5× single run |
| 3 | DQN (SB3) | No | ~5× single run |
| 4 | Masked Rec PPO | Yes | ~5× single run |
| 5 | Rec PPO No Mask | No | ~5× single run |
| 6 | PPO (SB3) | No | ~5× single run |

Each trained for **100,000 timesteps**, 5 seeds each = 30 training runs total.

**Baselines** (Random, Greedy, Entropy-Weighted) are re-run 5 times with different seeds as well (these are fast).

All per-episode `normalized_return` values are saved to CSV for downstream plotting.

In [ ]:
import random
import time
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from torch.utils.tensorboard import SummaryWriter
from torch.distributions import Categorical
from tqdm import tqdm
import pickle
import json
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

from Env import VoltageDistributionEnv
from CustomFeatureExtractor import CustomFeatureExtractor
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

import sys
sys.path.insert(0, os.path.abspath('../environment_utils'))

with open('../configuration/grid_configuration.json', 'r') as f:
    grid_config = json.load(f)
grid_size = grid_config['gridSize']

# Training configuration 
NUM_SEEDS = 5
SEEDS = [42, 123, 256, 789, 1024]
TRAINING_STEPS = 100_000
GRID_TAG = f"{grid_size}x{grid_size}"  # e.g. "11x11"

# Output directory for multi-seed CSVs
OUTPUT_DIR = os.path.join("..", "PlotsInPaper", "MultiSeed", GRID_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Grid: {GRID_TAG}")
print(f"Seeds: {SEEDS}")
print(f"Training steps per run: {TRAINING_STEPS:,}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def set_all_seeds(seed):
    """Set seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class CSVLogger:
    """
    Logs (episode_index, normalized_return) to a CSV matching
    the TensorBoard export format: Wall time, Step, Value
    """
    def __init__(self, filepath):
        self.filepath = filepath
        self.rows = []

    def log(self, step, value):
        self.rows.append({"Wall time": time.time(), "Step": step, "Value": value})

    def save(self):
        df = pd.DataFrame(self.rows)
        df.to_csv(self.filepath, index=False)
        print(f"  Saved {len(self.rows)} episodes -> {self.filepath}")

## Part 1: Baselines (Random, Greedy, Entropy-Weighted)

These are fast inference-only loops (~2200 episodes each). We run each 5 times and save CSVs.

In [ ]:
NUM_BASELINE_EPISODES = 2200

for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"Random_{GRID_TAG}_seed{seed_idx}.csv"))

    for ep in tqdm(range(NUM_BASELINE_EPISODES), desc=f"Random seed={seed}"):
        env = VoltageDistributionEnv()
        obs, info = env.reset()
        done, total_reward, steps = False, 0, 0
        while not done:
            action = np.random.randint(0, grid_size * grid_size)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1
        csv_log.log(ep, total_reward / steps)
        env.close()

    csv_log.save()

print("Random baselines complete")

In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"Greedy_{GRID_TAG}_seed{seed_idx}.csv"))

    for ep in tqdm(range(NUM_BASELINE_EPISODES), desc=f"Greedy seed={seed}"):
        env = VoltageDistributionEnv()
        obs, info = env.reset()
        done, total_reward, steps = False, 0, 0
        while not done:
            saved_belief = env.belief.belief.copy()
            env.belief.prediction()
            predicted_belief = env.belief.get_belief_distribution()
            env.belief.belief = saved_belief
            action = np.argmax(predicted_belief)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1
        csv_log.log(ep, total_reward / steps)
        env.close()

    csv_log.save()

print("Greedy baselines complete")

In [ ]:
def compute_local_entropy(belief, grid_size, window=3):
    """Compute local entropy for each position using a sliding window."""
    belief_2d = belief.reshape(grid_size, grid_size)
    local_entropy = np.zeros_like(belief_2d)
    half_w = window // 2
    for i in range(grid_size):
        for j in range(grid_size):
            i_min, i_max = max(0, i - half_w), min(grid_size, i + half_w + 1)
            j_min, j_max = max(0, j - half_w), min(grid_size, j + half_w + 1)
            local_probs = belief_2d[i_min:i_max, j_min:j_max].flatten()
            local_sum = local_probs.sum()
            if local_sum > 1e-10:
                local_probs = local_probs / local_sum
                local_probs = local_probs[local_probs > 1e-10]
                local_entropy[i, j] = -np.sum(local_probs * np.log(local_probs))
            else:
                local_entropy[i, j] = 0
    return local_entropy.flatten()


for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"EntropyWeighted_{GRID_TAG}_seed{seed_idx}.csv"))

    for ep in tqdm(range(NUM_BASELINE_EPISODES), desc=f"Entropy-Weighted seed={seed}"):
        env = VoltageDistributionEnv()
        obs, info = env.reset()
        done, total_reward, steps = False, 0, 0
        while not done:
            saved_belief = env.belief.belief.copy()
            env.belief.prediction()
            predicted_belief = env.belief.get_belief_distribution()
            env.belief.belief = saved_belief
            local_entropy = compute_local_entropy(predicted_belief, grid_size)
            sampling_weights = local_entropy + 1e-10
            sampling_weights /= sampling_weights.sum()
            action = np.random.choice(grid_size * grid_size, p=sampling_weights)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            steps += 1
        csv_log.log(ep, total_reward / steps)
        env.close()

    csv_log.save()

print("Entropy-Weighted baselines complete")

## Part 2: SB3 DQN & PPO (non-recurrent baselines)

Standard Stable-Baselines3 DQN and PPO, 100k timesteps each, 5 seeds.

In [ ]:
class NormalizedReturnCallback(BaseCallback):
    """Logs normalized_return per episode to a CSVLogger."""
    def __init__(self, csv_logger, verbose=0):
        super().__init__(verbose)
        self.csv_logger = csv_logger
        self.episode_count = 0

    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                normalized_return = info['episode']['r'] / info['episode']['l']
                self.csv_logger.log(self.episode_count, normalized_return)
                self.episode_count += 1
        return True


for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"DQN_{GRID_TAG}_seed{seed_idx}.csv"))

    dqn_env = DummyVecEnv([lambda: Monitor(VoltageDistributionEnv())])
    dqn = DQN(
        "MultiInputPolicy",
        dqn_env,
        policy_kwargs=dict(
            features_extractor_class=CustomFeatureExtractor,
            features_extractor_kwargs=dict(features_dim=128)
        ),
        learning_rate=1e-4,
        buffer_size=1000,
        batch_size=32,
        gamma=0.99,
        exploration_fraction=0.05,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.001,
        target_update_interval=500,
        seed=seed,
        verbose=0,
    )

    print(f"\n--- DQN seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ---")
    dqn.learn(total_timesteps=TRAINING_STEPS, callback=NormalizedReturnCallback(csv_log))
    csv_log.save()
    dqn_env.close()
    del dqn

print("\nSB3 DQN multi-seed training complete")


--- DQN seed 42 (1/5) ---
  Saved 3255 episodes -> PlotsInPaper/MultiSeed/11x11/DQN_11x11_seed0.csv

--- DQN seed 123 (2/5) ---
  Saved 3260 episodes -> PlotsInPaper/MultiSeed/11x11/DQN_11x11_seed1.csv

--- DQN seed 256 (3/5) ---
  Saved 3261 episodes -> PlotsInPaper/MultiSeed/11x11/DQN_11x11_seed2.csv

--- DQN seed 789 (4/5) ---
  Saved 3247 episodes -> PlotsInPaper/MultiSeed/11x11/DQN_11x11_seed3.csv

--- DQN seed 1024 (5/5) ---
  Saved 3258 episodes -> PlotsInPaper/MultiSeed/11x11/DQN_11x11_seed4.csv

SB3 DQN multi-seed training complete!


In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"PPO_{GRID_TAG}_seed{seed_idx}.csv"))

    ppo_env = DummyVecEnv([lambda: Monitor(VoltageDistributionEnv())])
    ppo = PPO(
        "MultiInputPolicy",
        ppo_env,
        policy_kwargs=dict(
            features_extractor_class=CustomFeatureExtractor,
            features_extractor_kwargs=dict(features_dim=128)
        ),
        learning_rate=1e-4,
        n_steps=128,
        batch_size=32,
        gamma=0.99,
        seed=seed,
        verbose=0,
    )

    print(f"\n--- PPO seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ---")
    ppo.learn(total_timesteps=TRAINING_STEPS, callback=NormalizedReturnCallback(csv_log))
    csv_log.save()
    ppo_env.close()
    del ppo

print("\nSB3 PPO multi-seed training complete")


--- PPO seed 42 (1/5) ---
  Saved 3257 episodes -> PlotsInPaper/MultiSeed/11x11/PPO_11x11_seed0.csv

--- PPO seed 123 (2/5) ---
  Saved 3263 episodes -> PlotsInPaper/MultiSeed/11x11/PPO_11x11_seed1.csv

--- PPO seed 256 (3/5) ---
  Saved 3264 episodes -> PlotsInPaper/MultiSeed/11x11/PPO_11x11_seed2.csv

--- PPO seed 789 (4/5) ---
  Saved 3250 episodes -> PlotsInPaper/MultiSeed/11x11/PPO_11x11_seed3.csv

--- PPO seed 1024 (5/5) ---
  Saved 3262 episodes -> PlotsInPaper/MultiSeed/11x11/PPO_11x11_seed4.csv

SB3 PPO multi-seed training complete!


## Part 3: Recurrent DQN (custom DRQN)

Masked and unmasked Recurrent DQN, 100k timesteps, 5 seeds each.

In [ ]:
class DRQNagent:
    """
    Deep Recurrent Q-Network — identical to RecurrentDQN.ipynb
    but with seed support and CSV logging for multi-seed runs.
    """

    def __init__(self, env, feature_extractor, action_dim,
                 lstm_hidden_size=256, gamma=0.99, lr=1e-4,
                 batch_size=32, buffer_size=1000,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=5000,
                 target_update=500, training_steps=10_000,
                 tensorboard_dir=None, run_name=None,
                 action_mask=False, csv_logger=None):

        self.env = env()
        self.feature_extractor_fn = feature_extractor
        self.action_dim = action_dim
        self.lstm_hidden_size = lstm_hidden_size
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.buffer_size = buffer_size
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.target_update = target_update
        self.training_steps = training_steps
        self.action_mask = action_mask
        self.csv_logger = csv_logger
        self.device = torch.device('cpu')

        # TensorBoard (optional — can skip for multi-seed runs)
        if tensorboard_dir and run_name:
            log_dir = os.path.join(tensorboard_dir, run_name)
            self.writer = SummaryWriter(log_dir=log_dir)
        else:
            self.writer = None

        feature_extractor_inst = feature_extractor(self.env.observation_space)
        feat_dim = feature_extractor_inst._features_dim

        self.online_net = self._build_network(feature_extractor_inst, feat_dim, lstm_hidden_size, action_dim)
        self.target_net = self._build_network(self.feature_extractor_fn(self.env.observation_space), feat_dim, lstm_hidden_size, action_dim)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=lr)
        self.replay_buffer = deque(maxlen=buffer_size)
        self.step_count = 0
        self.update_count = 0
        self.episode_count = 0

    def _build_network(self, feat_extractor, feat_dim, hidden_size, action_dim):
        class DRQNNet(nn.Module):
            def __init__(self, feat_extractor, feat_dim, hidden_size, action_dim):
                super().__init__()
                self.feat_extractor = feat_extractor
                self.lstm = nn.LSTM(feat_dim, hidden_size, batch_first=True)
                self.fc = nn.Sequential(
                    nn.Linear(hidden_size, 256), nn.ReLU(),
                    nn.Linear(256, action_dim)
                )
            def forward(self, obs_dict, hidden=None):
                B = obs_dict[list(obs_dict.keys())[0]].shape[0]
                T = obs_dict[list(obs_dict.keys())[0]].shape[1]
                flat_obs = {k: v.reshape(B*T, *v.shape[2:]) for k, v in obs_dict.items()}
                features = self.feat_extractor(flat_obs)
                features = features.view(B, T, -1)
                lstm_out, hidden = self.lstm(features, hidden)
                q_values = self.fc(lstm_out)
                return q_values, hidden
        return DRQNNet(feat_extractor, feat_dim, hidden_size, action_dim)

    def get_epsilon(self):
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * np.exp(-self.step_count / self.epsilon_decay)

    def select_action(self, obs_dict, info, hidden, epsilon):
        mask = None
        if self.action_mask:
            mask = info.get("action_mask", None)
        if random.random() < epsilon:
            if mask is None:
                action = self.env.action_space.sample()
            else:
                valid_actions = np.where(np.array(mask) == 1)[0]
                action = np.random.choice(valid_actions)
            return action, hidden
        with torch.no_grad():
            obs_batch = {k: torch.from_numpy(v).unsqueeze(0).unsqueeze(0).float().to(self.device) for k, v in obs_dict.items()}
            q_vals, new_hidden = self.online_net(obs_batch, hidden)
            q_vals = q_vals.squeeze(0).squeeze(0)
            if mask is not None:
                mask_tensor = torch.from_numpy(mask).to(self.device)
                q_vals = q_vals.masked_fill(mask_tensor == 0, -1e9)
            action = q_vals.argmax().item()
        return action, new_hidden

    def store_episode(self, episode):
        self.replay_buffer.append(episode)

    def sample_batch(self):
        if len(self.replay_buffer) < self.batch_size:
            return None
        return random.sample(self.replay_buffer, self.batch_size)

    def train_step(self):
        batch = self.sample_batch()
        if batch is None:
            return None
        total_loss = 0
        for episode in batch:
            obs_seq, action_seq, reward_seq, next_obs_seq, done_seq, mask_seq = [], [], [], [], [], []
            for transition in episode:
                if len(transition) == 6:
                    obs, action, reward, next_obs, done, info = transition
                    mask_seq.append(info.get("action_mask", None))
                else:
                    obs, action, reward, next_obs, done = transition
                    mask_seq.append(None)
                obs_seq.append(obs)
                action_seq.append(action)
                reward_seq.append(reward)
                next_obs_seq.append(next_obs)
                done_seq.append(done)

            obs_batch = {k: torch.from_numpy(np.stack([o[k] for o in obs_seq])).unsqueeze(0).float().to(self.device)
                         for k in obs_seq[0].keys()}
            next_obs_batch = {k: torch.from_numpy(np.stack([o[k] for o in next_obs_seq])).unsqueeze(0).float().to(self.device)
                              for k in next_obs_seq[0].keys()}
            actions = torch.LongTensor(action_seq).unsqueeze(0).to(self.device)
            rewards = torch.FloatTensor(reward_seq).unsqueeze(0).to(self.device)
            dones = torch.FloatTensor(done_seq).unsqueeze(0).to(self.device)

            hidden = None
            q_values, hidden = self.online_net(obs_batch, hidden)
            q_values = q_values.squeeze(0)
            current_q = q_values.gather(1, actions.squeeze(0).unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                target_hidden = None
                next_q_values, target_hidden = self.target_net(next_obs_batch, target_hidden)
                next_q_values = next_q_values.squeeze(0)
                if self.action_mask:
                    assert all(m is not None for m in mask_seq)
                    mask_tensor = torch.from_numpy(np.stack(mask_seq)).to(self.device)
                    next_q_values = next_q_values.masked_fill(mask_tensor == 0, -1e9)
                max_next_q = next_q_values.max(dim=1)[0]
                target_q = rewards.squeeze(0) + self.gamma * max_next_q * (1 - dones.squeeze(0))

            loss = nn.functional.mse_loss(current_q, target_q)
            total_loss += loss.item()
            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.online_net.parameters(), 1.0)
            self.optimizer.step()
        return total_loss / len(batch)

    def train(self):
        episode_return_window = deque(maxlen=100)
        with tqdm(total=self.training_steps, desc="Training DRQN", unit="step") as pbar:
            while self.step_count < self.training_steps:
                obs, _ = self.env.reset()
                done = False
                info = {}
                episode = []
                episode_return = 0
                hidden = None
                episode_steps = 0

                while not done:
                    epsilon = self.get_epsilon()
                    action, hidden = self.select_action(obs, info, hidden, epsilon)
                    next_obs, reward, terminated, truncated, info = self.env.step(action)
                    done = terminated or truncated
                    episode.append((obs, action, reward, next_obs, done, info) if self.action_mask else (obs, action, reward, next_obs, done))
                    episode_return += reward
                    obs = next_obs
                    self.step_count += 1
                    episode_steps += 1

                pbar.update(episode_steps)
                self.store_episode(episode)
                self.episode_count += 1
                episode_return_window.append(episode_return)

                normalized_return = episode_return / episode_steps if episode_steps > 0 else 0

                # Log to CSV
                if self.csv_logger:
                    self.csv_logger.log(self.episode_count, normalized_return)

                # Log to TensorBoard (optional)
                if self.writer:
                    self.writer.add_scalar('rollout/normalized_return', normalized_return, self.episode_count)

                # Train step
                if len(self.replay_buffer) >= self.batch_size:
                    self.train_step()
                    self.update_count += 1
                    if self.update_count % self.target_update == 0:
                        self.target_net.load_state_dict(self.online_net.state_dict())

        if self.writer:
            self.writer.close()
        if self.csv_logger:
            self.csv_logger.save()
        print(f"Training complete: {self.episode_count} episodes, {self.step_count} steps")

In [ ]:
env_instance = VoltageDistributionEnv()
action_dim = env_instance.action_space.n
env_instance.close()

for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"MaskedDRQN_{GRID_TAG}_seed{seed_idx}.csv"))

    print(f"\n=== Masked DRQN seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ===")
    agent = DRQNagent(
        env=lambda: VoltageDistributionEnv(),
        feature_extractor=lambda obs_space: CustomFeatureExtractor(obs_space),
        action_dim=action_dim,
        lstm_hidden_size=256,
        gamma=0.9,
        lr=1e-4,
        batch_size=32,
        buffer_size=1_000,
        epsilon_start=1.0,
        epsilon_end=0.001,
        epsilon_decay=5_000,
        target_update=500,
        training_steps=TRAINING_STEPS,
        action_mask=True,
        csv_logger=csv_log,
    )
    agent.train()
    del agent

print("\nMasked DRQN multi-seed training complete!")


=== Masked DRQN seed 42 (1/5) ===


Training DRQN: 100024step [24:01, 69.40step/s]                         


  Saved 3244 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedDRQN_11x11_seed0.csv
Training complete: 3244 episodes, 100024 steps

=== Masked DRQN seed 123 (2/5) ===


Training DRQN: 100016step [24:14, 68.76step/s]                         


  Saved 3263 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedDRQN_11x11_seed1.csv
Training complete: 3263 episodes, 100016 steps

=== Masked DRQN seed 256 (3/5) ===


Training DRQN: 100020step [23:07, 72.10step/s]                         


  Saved 3257 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedDRQN_11x11_seed2.csv
Training complete: 3257 episodes, 100020 steps

=== Masked DRQN seed 789 (4/5) ===


Training DRQN: 100004step [24:21, 68.45step/s]                         


  Saved 3255 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedDRQN_11x11_seed3.csv
Training complete: 3255 episodes, 100004 steps

=== Masked DRQN seed 1024 (5/5) ===


Training DRQN: 100018step [2:35:52, 10.69step/s]                             

  Saved 3261 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedDRQN_11x11_seed4.csv
Training complete: 3261 episodes, 100018 steps

Masked DRQN multi-seed training complete!


In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"DRQNNoMask_{GRID_TAG}_seed{seed_idx}.csv"))

    print(f"\n=== DRQN No Mask seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ===")
    agent = DRQNagent(
        env=lambda: VoltageDistributionEnv(),
        feature_extractor=lambda obs_space: CustomFeatureExtractor(obs_space),
        action_dim=action_dim,
        lstm_hidden_size=256,
        gamma=0.9,
        lr=1e-4,
        batch_size=32,
        buffer_size=1_000,
        epsilon_start=1.0,
        epsilon_end=0.001,
        epsilon_decay=5_000,
        target_update=500,
        training_steps=TRAINING_STEPS,
        action_mask=False,
        csv_logger=csv_log,
    )
    agent.train()
    del agent

print("\nDRQN No Mask multi-seed training complete")


=== DRQN No Mask seed 42 (1/5) ===


Training DRQN: 100024step [23:08, 72.03step/s]                         


  Saved 3244 episodes -> PlotsInPaper/MultiSeed/11x11/DRQNNoMask_11x11_seed0.csv
Training complete: 3244 episodes, 100024 steps

=== DRQN No Mask seed 123 (2/5) ===


Training DRQN: 100016step [25:15, 66.01step/s]                         


  Saved 3263 episodes -> PlotsInPaper/MultiSeed/11x11/DRQNNoMask_11x11_seed1.csv
Training complete: 3263 episodes, 100016 steps

=== DRQN No Mask seed 256 (3/5) ===


Training DRQN: 100020step [25:49, 64.53step/s]                         


  Saved 3257 episodes -> PlotsInPaper/MultiSeed/11x11/DRQNNoMask_11x11_seed2.csv
Training complete: 3257 episodes, 100020 steps

=== DRQN No Mask seed 789 (4/5) ===


Training DRQN: 100004step [25:30, 65.35step/s]                         


  Saved 3255 episodes -> PlotsInPaper/MultiSeed/11x11/DRQNNoMask_11x11_seed3.csv
Training complete: 3255 episodes, 100004 steps

=== DRQN No Mask seed 1024 (5/5) ===


Training DRQN: 100018step [26:07, 63.81step/s]                         

  Saved 3261 episodes -> PlotsInPaper/MultiSeed/11x11/DRQNNoMask_11x11_seed4.csv
Training complete: 3261 episodes, 100018 steps

DRQN No Mask multi-seed training complete!


## Part 4: Recurrent PPO

Masked and unmasked Recurrent PPO (LSTM), 100k timesteps, 5 seeds each.

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    if layer.bias is not None:
        torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class RecurrentPPOAgent:
    """
    Recurrent PPO with LSTM — identical to RecurrentPPO.ipynb
    but with seed support and CSV logging for multi-seed runs.
    """

    def __init__(self, env_constructor, feature_extractor_constructor, action_dim,
                 lstm_hidden_size=256, lstm_num_layers=1,
                 gamma=0.99, gae_lambda=0.95, lr=1e-4,
                 clip_coef=0.2, vf_coef=0.5, ent_coef=0.01,
                 max_grad_norm=0.5, update_epochs=4, num_minibatches=4,
                 norm_adv=True, clip_vloss=True,
                 num_envs=4, num_steps=128, training_steps=10_000,
                 tensorboard_dir=None, run_name=None,
                 action_mask=False, csv_logger=None):

        self.env_constructor = env_constructor
        self.feature_extractor_constructor = feature_extractor_constructor
        self.action_dim = action_dim
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.lr = lr
        self.clip_coef = clip_coef
        self.vf_coef = vf_coef
        self.ent_coef = ent_coef
        self.max_grad_norm = max_grad_norm
        self.update_epochs = update_epochs
        self.num_minibatches = num_minibatches
        self.norm_adv = norm_adv
        self.clip_vloss = clip_vloss
        self.num_envs = num_envs
        self.num_steps = num_steps
        self.training_steps = training_steps
        self.action_mask_enabled = action_mask
        self.csv_logger = csv_logger
        self.device = torch.device('cpu')

        single_env = env_constructor()
        self.observation_space = single_env.observation_space
        single_env.close()

        self.agent = self._build_agent()
        self.optimizer = optim.Adam(self.agent.parameters(), lr=lr)

        if tensorboard_dir and run_name:
            log_dir = os.path.join(tensorboard_dir, run_name)
            self.writer = SummaryWriter(log_dir=log_dir)
        else:
            self.writer = None

        self.global_step = 0
        self.episode_count = 0
        self.update_count = 0
        self.batch_size = num_envs * num_steps
        self.minibatch_size = self.batch_size // num_minibatches

    def _build_agent(self):
        class PPOLSTMAgent(nn.Module):
            def __init__(self, parent):
                super().__init__()
                self.parent = parent
                self.feature_extractor = parent.feature_extractor_constructor(parent.observation_space)
                feat_dim = self.feature_extractor._features_dim
                self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=parent.lstm_hidden_size,
                                    num_layers=parent.lstm_num_layers, batch_first=True)
                for name, param in self.lstm.named_parameters():
                    if 'weight_ih' in name: nn.init.orthogonal_(param)
                    elif 'weight_hh' in name: nn.init.orthogonal_(param)
                    elif 'bias' in name: nn.init.zeros_(param)
                self.actor = nn.Sequential(
                    layer_init(nn.Linear(parent.lstm_hidden_size, parent.lstm_hidden_size)),
                    nn.ReLU(),
                    layer_init(nn.Linear(parent.lstm_hidden_size, parent.action_dim), std=0.01)
                )
                self.critic = nn.Sequential(
                    layer_init(nn.Linear(parent.lstm_hidden_size, parent.lstm_hidden_size)),
                    nn.ReLU(),
                    layer_init(nn.Linear(parent.lstm_hidden_size, 1), std=1.0)
                )

            def get_initial_lstm_state(self, batch_size):
                h0 = torch.zeros(self.parent.lstm_num_layers, batch_size, self.parent.lstm_hidden_size).to(self.parent.device)
                c0 = torch.zeros(self.parent.lstm_num_layers, batch_size, self.parent.lstm_hidden_size).to(self.parent.device)
                return (h0, c0)

            def get_value(self, obs_dict, lstm_state):
                features = self.feature_extractor(obs_dict)
                features = features.unsqueeze(1)
                lstm_out, new_lstm_state = self.lstm(features, lstm_state)
                lstm_out = lstm_out.squeeze(1)
                return self.critic(lstm_out).flatten(), new_lstm_state

            def get_action_and_value(self, obs_dict, lstm_state, action=None, action_mask=None):
                features = self.feature_extractor(obs_dict)
                features = features.unsqueeze(1)
                lstm_out, new_lstm_state = self.lstm(features, lstm_state)
                lstm_out = lstm_out.squeeze(1)
                logits = self.actor(lstm_out)
                if action_mask is not None:
                    logits = logits.masked_fill(action_mask == 0, float('-1e8'))
                dist = Categorical(logits=logits)
                if action is None:
                    action = dist.sample()
                return action, dist.log_prob(action), dist.entropy(), self.critic(lstm_out).flatten(), new_lstm_state

        return PPOLSTMAgent(self).to(self.device)

    def _make_vec_env(self):
        def make_env(idx):
            def thunk():
                env = self.env_constructor()
                env = gym.wrappers.RecordEpisodeStatistics(env)
                return env
            return thunk
        return gym.vector.SyncVectorEnv([make_env(i) for i in range(self.num_envs)])

    def _obs_dict_to_tensor(self, obs_dict, device):
        return {k: torch.from_numpy(v).float().to(device) for k, v in obs_dict.items()}

    def train(self):
        envs = self._make_vec_env()

        obs_buffer = {k: torch.zeros((self.num_steps, self.num_envs, *v.shape)).to(self.device)
                      for k, v in self.observation_space.spaces.items()}
        actions = torch.zeros((self.num_steps, self.num_envs), dtype=torch.long).to(self.device)
        log_probs = torch.zeros((self.num_steps, self.num_envs)).to(self.device)
        rewards = torch.zeros((self.num_steps, self.num_envs)).to(self.device)
        dones = torch.zeros((self.num_steps, self.num_envs)).to(self.device)
        values = torch.zeros((self.num_steps, self.num_envs)).to(self.device)
        action_masks_buffer = torch.ones((self.num_steps, self.num_envs, self.action_dim)).to(self.device)
        lstm_h_buffer = torch.zeros((self.num_steps, self.lstm_num_layers, self.num_envs, self.lstm_hidden_size)).to(self.device)
        lstm_c_buffer = torch.zeros((self.num_steps, self.lstm_num_layers, self.num_envs, self.lstm_hidden_size)).to(self.device)

        next_obs_np, infos = envs.reset()
        next_obs = self._obs_dict_to_tensor(next_obs_np, self.device)
        next_done = torch.zeros(self.num_envs).to(self.device)
        next_lstm_state = self.agent.get_initial_lstm_state(self.num_envs)

        next_action_mask = None
        if self.action_mask_enabled:
            next_action_mask = torch.ones((self.num_envs, self.action_dim)).to(self.device)
            if 'action_mask' in infos:
                for i in range(self.num_envs):
                    if infos['action_mask'][i] is not None:
                        next_action_mask[i] = torch.from_numpy(np.array(infos['action_mask'][i])).float().to(self.device)

        episode_return_window = deque(maxlen=100)
        episode_normalized_return_window = deque(maxlen=100)
        num_iterations = self.training_steps // self.batch_size + 1

        with tqdm(total=self.training_steps, desc="Training RecurrentPPO (LSTM)", unit="step") as pbar:
            for iteration in range(1, num_iterations + 1):
                if self.global_step >= self.training_steps:
                    break

                for step in range(self.num_steps):
                    if self.global_step >= self.training_steps:
                        break
                    self.global_step += self.num_envs

                    with torch.no_grad():
                        for k in next_obs.keys():
                            obs_buffer[k][step] = next_obs[k]
                        dones[step] = next_done
                        lstm_h_buffer[step] = next_lstm_state[0]
                        lstm_c_buffer[step] = next_lstm_state[1]
                        if self.action_mask_enabled and next_action_mask is not None:
                            action_masks_buffer[step] = next_action_mask
                        action, log_prob, _, value, new_lstm_state = self.agent.get_action_and_value(
                            next_obs, next_lstm_state,
                            action_mask=next_action_mask if self.action_mask_enabled else None
                        )
                        next_lstm_state = new_lstm_state
                        actions[step] = action
                        log_probs[step] = log_prob
                        values[step] = value

                    next_obs_np, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
                    next_done_np = np.logical_or(terminated, truncated)
                    rewards[step] = torch.tensor(reward).to(self.device)
                    next_obs = self._obs_dict_to_tensor(next_obs_np, self.device)
                    next_done = torch.tensor(next_done_np).float().to(self.device)

                    if self.action_mask_enabled:
                        next_action_mask = torch.ones((self.num_envs, self.action_dim)).to(self.device)
                        if 'action_mask' in infos:
                            for i in range(self.num_envs):
                                mask = infos['action_mask'][i] if isinstance(infos['action_mask'], (list, np.ndarray)) else infos['action_mask']
                                if mask is not None:
                                    next_action_mask[i] = torch.from_numpy(np.array(mask)).float().to(self.device)

                    for idx, done_flag in enumerate(next_done_np):
                        if done_flag:
                            next_lstm_state[0][:, idx, :] = 0
                            next_lstm_state[1][:, idx, :] = 0

                    if 'episode' in infos and isinstance(infos['episode'], dict):
                        ep_data = infos['episode']
                        for i in range(self.num_envs):
                            if next_done_np[i]:
                                ep_return = float(ep_data['r'][i])
                                ep_length = int(ep_data['l'][i])
                                self.episode_count += 1
                                normalized_return = ep_return / ep_length if ep_length > 0 else 0
                                episode_return_window.append(ep_return)
                                episode_normalized_return_window.append(normalized_return)

                                # Log to CSV
                                if self.csv_logger:
                                    self.csv_logger.log(self.episode_count, normalized_return)

                                if self.writer:
                                    self.writer.add_scalar('rollout/normalized_return', normalized_return, self.episode_count)

                    pbar.update(self.num_envs)

                # Bootstrap + GAE
                with torch.no_grad():
                    next_value, _ = self.agent.get_value(next_obs, next_lstm_state)
                    advantages = torch.zeros_like(rewards).to(self.device)
                    lastgaelam = 0
                    for t in reversed(range(self.num_steps)):
                        if t == self.num_steps - 1:
                            nextnonterminal = 1.0 - next_done
                            nextvalues = next_value
                        else:
                            nextnonterminal = 1.0 - dones[t + 1]
                            nextvalues = values[t + 1]
                        delta = rewards[t] + self.gamma * nextvalues * nextnonterminal - values[t]
                        advantages[t] = lastgaelam = delta + self.gamma * self.gae_lambda * nextnonterminal * lastgaelam
                    returns = advantages + values

                # Flatten
                b_obs = {k: v.reshape(-1, *v.shape[2:]) for k, v in obs_buffer.items()}
                b_log_probs = log_probs.reshape(-1)
                b_actions = actions.reshape(-1)
                b_advantages = advantages.reshape(-1)
                b_returns = returns.reshape(-1)
                b_values = values.reshape(-1)
                b_action_masks = action_masks_buffer.reshape(-1, self.action_dim)
                b_lstm_h = lstm_h_buffer.permute(0, 2, 1, 3).reshape(-1, self.lstm_num_layers, self.lstm_hidden_size)
                b_lstm_c = lstm_c_buffer.permute(0, 2, 1, 3).reshape(-1, self.lstm_num_layers, self.lstm_hidden_size)

                # PPO update
                for epoch in range(self.update_epochs):
                    b_inds = torch.randperm(self.batch_size)
                    for start in range(0, self.batch_size, self.minibatch_size):
                        end = start + self.minibatch_size
                        mb_inds = b_inds[start:end]
                        mb_obs = {k: v[mb_inds] for k, v in b_obs.items()}
                        mb_lstm_h = b_lstm_h[mb_inds].permute(1, 0, 2).contiguous()
                        mb_lstm_c = b_lstm_c[mb_inds].permute(1, 0, 2).contiguous()
                        mb_lstm_state = (mb_lstm_h, mb_lstm_c)

                        _, newlogprob, entropy, newvalue, _ = self.agent.get_action_and_value(
                            mb_obs, mb_lstm_state,
                            action=b_actions[mb_inds],
                            action_mask=b_action_masks[mb_inds] if self.action_mask_enabled else None
                        )

                        mb_advantages = b_advantages[mb_inds]
                        if self.norm_adv:
                            mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                        logratio = newlogprob - b_log_probs[mb_inds]
                        ratio = torch.exp(logratio)

                        pg_loss1 = -mb_advantages * ratio
                        pg_loss2 = -mb_advantages * torch.clamp(ratio, 1.0 - self.clip_coef, 1.0 + self.clip_coef)
                        pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                        returns_min = b_returns.min()
                        returns_max = b_returns.max()
                        den = returns_max - returns_min + 1e-8
                        normalized_returns = (b_returns - returns_min) / den
                        normalized_values = (newvalue - returns_min) / den

                        if self.clip_vloss:
                            v_loss_unclipped = (normalized_values - normalized_returns[mb_inds]) ** 2
                            v_clipped = ((b_values[mb_inds] - returns_min) / den) + torch.clamp(
                                normalized_values - ((b_values[mb_inds] - returns_min) / den),
                                -self.clip_coef, self.clip_coef
                            )
                            v_loss_clipped = (v_clipped - normalized_returns[mb_inds]) ** 2
                            v_loss = torch.max(v_loss_unclipped, v_loss_clipped).mean()
                        else:
                            v_loss = ((normalized_values - normalized_returns[mb_inds]) ** 2).mean()

                        entropy_loss = entropy.mean() / np.log(self.action_dim)
                        loss = pg_loss - (self.ent_coef * entropy_loss) + (self.vf_coef * v_loss)

                        self.optimizer.zero_grad()
                        loss.backward()
                        nn.utils.clip_grad_norm_(self.agent.parameters(), self.max_grad_norm)
                        self.optimizer.step()
                        self.update_count += 1

        envs.close()
        if self.writer:
            self.writer.close()
        if self.csv_logger:
            self.csv_logger.save()
        print(f"Training complete: {self.episode_count} episodes, {self.global_step} steps")

In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"MaskedRecPPO_{GRID_TAG}_seed{seed_idx}.csv"))

    print(f"\n=== Masked Rec PPO seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ===")
    agent = RecurrentPPOAgent(
        env_constructor=lambda: VoltageDistributionEnv(),
        feature_extractor_constructor=lambda obs_space: CustomFeatureExtractor(obs_space),
        action_dim=action_dim,
        lstm_hidden_size=256,
        lstm_num_layers=1,
        gamma=0.99,
        gae_lambda=0.95,
        lr=1e-4,
        clip_coef=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        max_grad_norm=0.5,
        update_epochs=4,
        num_minibatches=4,
        norm_adv=True,
        clip_vloss=True,
        num_envs=4,
        num_steps=128,
        training_steps=TRAINING_STEPS,
        action_mask=True,
        csv_logger=csv_log,
    )
    agent.train()
    del agent

print("\nMasked Rec PPO multi-seed training complete")


=== Masked Rec PPO seed 42 (1/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:05<00:00, 798.05step/s]


  Saved 3154 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedRecPPO_11x11_seed0.csv
Training complete: 3154 episodes, 100000 steps

=== Masked Rec PPO seed 123 (2/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:05<00:00, 798.70step/s]


  Saved 3157 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedRecPPO_11x11_seed1.csv
Training complete: 3157 episodes, 100000 steps

=== Masked Rec PPO seed 256 (3/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:01<00:00, 824.94step/s]


  Saved 3158 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedRecPPO_11x11_seed2.csv
Training complete: 3158 episodes, 100000 steps

=== Masked Rec PPO seed 789 (4/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:03<00:00, 810.21step/s]


  Saved 3143 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedRecPPO_11x11_seed3.csv
Training complete: 3143 episodes, 100000 steps

=== Masked Rec PPO seed 1024 (5/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [01:52<00:00, 888.36step/s]

  Saved 3155 episodes -> PlotsInPaper/MultiSeed/11x11/MaskedRecPPO_11x11_seed4.csv
Training complete: 3155 episodes, 100000 steps

Masked Rec PPO multi-seed training complete!


In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    set_all_seeds(seed)
    csv_log = CSVLogger(os.path.join(OUTPUT_DIR, f"RecPPONoMask_{GRID_TAG}_seed{seed_idx}.csv"))

    print(f"\n=== Rec PPO No Mask seed {seed} ({seed_idx+1}/{NUM_SEEDS}) ===")
    agent = RecurrentPPOAgent(
        env_constructor=lambda: VoltageDistributionEnv(),
        feature_extractor_constructor=lambda obs_space: CustomFeatureExtractor(obs_space),
        action_dim=action_dim,
        lstm_hidden_size=256,
        lstm_num_layers=1,
        gamma=0.99,
        gae_lambda=0.95,
        lr=1e-4,
        clip_coef=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        max_grad_norm=0.5,
        update_epochs=4,
        num_minibatches=4,
        norm_adv=True,
        clip_vloss=True,
        num_envs=4,
        num_steps=128,
        training_steps=TRAINING_STEPS,
        action_mask=False,
        csv_logger=csv_log,
    )
    agent.train()
    del agent

print("\nRec PPO No Mask multi-seed training complete")


=== Rec PPO No Mask seed 42 (1/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:07<00:00, 786.18step/s]


  Saved 3154 episodes -> PlotsInPaper/MultiSeed/11x11/RecPPONoMask_11x11_seed0.csv
Training complete: 3154 episodes, 100000 steps

=== Rec PPO No Mask seed 123 (2/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:00<00:00, 832.36step/s]


  Saved 3157 episodes -> PlotsInPaper/MultiSeed/11x11/RecPPONoMask_11x11_seed1.csv
Training complete: 3157 episodes, 100000 steps

=== Rec PPO No Mask seed 256 (3/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [02:04<00:00, 805.54step/s]


  Saved 3158 episodes -> PlotsInPaper/MultiSeed/11x11/RecPPONoMask_11x11_seed2.csv
Training complete: 3158 episodes, 100000 steps

=== Rec PPO No Mask seed 789 (4/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [01:55<00:00, 863.87step/s]


  Saved 3143 episodes -> PlotsInPaper/MultiSeed/11x11/RecPPONoMask_11x11_seed3.csv
Training complete: 3143 episodes, 100000 steps

=== Rec PPO No Mask seed 1024 (5/5) ===


Training RecurrentPPO (LSTM): 100%|██████████| 100000/100000 [01:55<00:00, 868.75step/s]

  Saved 3155 episodes -> PlotsInPaper/MultiSeed/11x11/RecPPONoMask_11x11_seed4.csv
Training complete: 3155 episodes, 100000 steps

Rec PPO No Mask multi-seed training complete!


## Part 5: Statistical Significance testing

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

GRID_TAG = "11x11"
OUTPUT_DIR = os.path.join("..", "PlotsInPaper", "MultiSeed", GRID_TAG)
NUM_SEEDS = 5
TAIL_FRAC = 0.10  # last 10% of episodes

methods = {
    "MaskedDRQN":       "Masked DRQN",
    "DRQNNoMask":       "DRQN (No Mask)",
    "DQN":              "DQN",
    "MaskedRecPPO":     "Masked Recurrent PPO",
    "RecPPONoMask":     "Recurrent PPO (No Mask)",
    "PPO":              "PPO",
    "EntropyWeighted":  "Entropy Weighted",
    "Greedy":           "Greedy",
    "Random":           "Random",
}

def get_final_performance(prefix, grid_tag, output_dir, num_seeds, tail_frac):
    """For each seed, compute avg reward over last tail_frac of episodes."""
    seed_finals = []
    for s in range(num_seeds):
        fpath = os.path.join(output_dir, f"{prefix}_{grid_tag}_seed{s}.csv")
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath).sort_values("Step")
        n = len(df)
        tail = df.iloc[int(n * (1 - tail_frac)):]
        seed_finals.append(tail["Value"].mean())
    return np.array(seed_finals)

# Build table rows
rows = []
for prefix, name in methods.items():
    finals = get_final_performance(prefix, GRID_TAG, OUTPUT_DIR, NUM_SEEDS, TAIL_FRAC)
    if len(finals) == 0:
        continue
    mean = finals.mean()
    std = finals.std(ddof=1)
    n = len(finals)
    ci95 = stats.t.ppf(0.975, df=n-1) * std / np.sqrt(n)
    rows.append({
        "Method": name,
        "Mean Reward": f"{mean:.4f}",
        "Std Dev": f"{std:.4f}",
        "95% CI": f"[{mean - ci95:.4f}, {mean + ci95:.4f}]",
        "_sort": mean,
    })

robustness_df = pd.DataFrame(rows).sort_values("_sort", ascending=False).drop(columns=["_sort"]).reset_index(drop=True)
robustness_df.index += 1
robustness_df.index.name = "Rank"

In [ ]:
REF_PREFIX = "MaskedDRQN"
REF_NAME = "Masked DRQN"

baselines = {
    "DRQNNoMask":       "DRQN (No Mask)",
    "DQN":              "DQN",
    "MaskedRecPPO":     "Masked Recurrent PPO",
    "RecPPONoMask":     "Recurrent PPO (No Mask)",
    "PPO":              "PPO",
    "EntropyWeighted":  "Entropy Weighted",
    "Greedy":           "Greedy",
    "Random":           "Random",
}

ref_finals = get_final_performance(REF_PREFIX, GRID_TAG, OUTPUT_DIR, NUM_SEEDS, TAIL_FRAC)

rows = []
for prefix, name in baselines.items():
    bl_finals = get_final_performance(prefix, GRID_TAG, OUTPUT_DIR, NUM_SEEDS, TAIL_FRAC)
    if len(bl_finals) == 0 or len(bl_finals) != len(ref_finals):
        continue
    diff = ref_finals - bl_finals
    t_stat, p_val = stats.ttest_rel(ref_finals, bl_finals)
    rows.append({
        "Baseline Method": name,
        "Masked DRQN Mean": f"{ref_finals.mean():.4f}",
        "Baseline Mean": f"{bl_finals.mean():.4f}",
        "Mean Difference": f"{diff.mean():.4f}",
        "t-statistic": f"{t_stat:.3f}",
        "p-value": f"{p_val:.4e}",
        "Significant": "Yes" if p_val < 0.05 else "No",
        "_sort": diff.mean(),
    })

comparison_df = pd.DataFrame(rows).sort_values("_sort", ascending=False).drop(columns=["_sort"]).reset_index(drop=True)
comparison_df.index += 1
